In [2]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set notebook display options
pd.set_option('display.max_columns', 150)
pd.set_option('display.max_rows', 100)
%matplotlib inline

# Define data paths based on project structure
RAW_DATA_DIR = os.path.join("..", "data", "raw")
PROCESSED_DATA_DIR = os.path.join("..", "data", "processed")

print(f"Checking raw data directory: {os.path.abspath(RAW_DATA_DIR)}")

Matplotlib is building the font cache; this may take a moment.


Checking raw data directory: c:\Users\Pythospach\Desktop\Football Manager\ai-service\data\raw


In [3]:
# Identify all expected CSV paths
file_mappings = {
    "male_players": os.path.join(RAW_DATA_DIR, "male_players.csv"),
    "female_players": os.path.join(RAW_DATA_DIR, "female_players.csv"),
    "male_teams": os.path.join(RAW_DATA_DIR, "male_teams.csv"),
    "female_teams": os.path.join(RAW_DATA_DIR, "female_teams.csv"),
    "male_coaches": os.path.join(RAW_DATA_DIR, "male_coaches.csv"),
    "female_coaches": os.path.join(RAW_DATA_DIR, "female_coaches.csv")
}

# Verify existence
for name, path in file_mappings.items():
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path) / (1024*1024):.2f} MB" if exists else "N/A"
    print(f"[{'✓' if exists else '✗'}] {name:<15} -> {path} ({size})")

[✓] male_players    -> ..\data\raw\male_players.csv (91.87 MB)
[✓] female_players  -> ..\data\raw\female_players.csv (2.41 MB)
[✓] male_teams      -> ..\data\raw\male_teams.csv (2.04 MB)
[✓] female_teams    -> ..\data\raw\female_teams.csv (0.07 MB)
[✓] male_coaches    -> ..\data\raw\male_coaches.csv (0.22 MB)
[✓] female_coaches  -> ..\data\raw\female_coaches.csv (0.01 MB)


In [4]:
def profile_file_structure(name, path):
    """Performs a preliminary shape and memory structural audit on a dataset."""
    if not os.path.exists(path):
        return None
    
    # Read first 100 rows to optimize typing check
    df_sample = pd.read_csv(path, nrows=100)
    
    # Read absolute structural metadata
    chunks = pd.read_csv(path, chunksize=10000, low_memory=False)
    total_rows = sum(len(chunk) for chunk in chunks)
    
    return {
        "Dataset": name,
        "Rows": total_rows,
        "Columns": len(df_sample.columns),
        "Memory Estimate": f"{os.path.getsize(path) / (1024*1024):.2f} MB"
    }

structural_profiles = []
for name, path in file_mappings.items():
    profile = profile_file_structure(name, path)
    if profile:
        structural_profiles.append(profile)

df_structure = pd.DataFrame(structural_profiles)
df_structure

,Dataset,Rows,Columns,Memory Estimate
0,male_players,180021,109,91.87 MB
1,female_players,5035,109,2.41 MB
2,male_teams,6947,54,2.04 MB
3,female_teams,231,54,0.07 MB
4,male_coaches,1369,8,0.22 MB
5,female_coaches,94,8,0.01 MB


In [5]:
print("=== AUDITING COMPOSITE SNAPSHOT KEYS ===")

def audit_keys_and_versioning(name, path, id_col):
    if not os.path.exists(path):
        return
        
    print(f"\nProcessing: {name}")
    df = pd.read_csv(path, low_memory=False)
    
    # Check if expected version columns actually exist
    version_cols = ['fifa_version', 'fifa_update', 'update_as_of']
    missing_v_cols = [c for c in version_cols if c not in df.columns]
    
    if missing_v_cols:
        print(f"  ⚠️ WARNING: Missing versioning metrics: {missing_v_cols}")
        # Dynamic fallback if coach data treats updates differently
        available_keys = [id_col] + [c for c in version_cols if c in df.columns]
    else:
        available_keys = [id_col, 'fifa_version', 'fifa_update']
        
    # Check uniqueness of the proposed Firestore composite snapshot key
    duplicate_count = df.duplicated(subset=available_keys).sum()
    print(f"  Target Composite Key: {available_keys}")
    print(f"  Duplicate snapshot signatures found: {duplicate_count} out of {len(df)} rows")
    
    if duplicate_count > 0:
        print(f"  ❌ Primary key violation! Snapshot pattern cannot guarantee identity for {name}.")
    else:
        print(f"  ✅ Perfect unique identification constraint met.")

# Execute across distinct entity branches
audit_keys_and_versioning("male_players", file_mappings["male_players"], "player_id")
audit_keys_and_versioning("male_teams", file_mappings["male_teams"], "team_id")
audit_keys_and_versioning("male_coaches", file_mappings["male_coaches"], "coach_id")

=== AUDITING COMPOSITE SNAPSHOT KEYS ===

Processing: male_players
  Target Composite Key: ['player_id', 'fifa_version', 'fifa_update']
  Duplicate snapshot signatures found: 0 out of 180021 rows
  ✅ Perfect unique identification constraint met.

Processing: male_teams
  Target Composite Key: ['team_id', 'fifa_version', 'fifa_update']
  Duplicate snapshot signatures found: 0 out of 6947 rows
  ✅ Perfect unique identification constraint met.

Processing: male_coaches
  ⚠️ WARNING: Missing versioning metrics: ['fifa_version', 'fifa_update', 'update_as_of']
  Target Composite Key: ['coach_id']
  Duplicate snapshot signatures found: 0 out of 1369 rows
  ✅ Perfect unique identification constraint met.


In [6]:
print("=== ANALYZING STRING-BASED MATHEMATICAL RATING ATTRIBUTES ===")

# Pick a target player sheet to audit positional notations (e.g., '82+2')
player_path = file_mappings["male_players"]

if os.path.exists(player_path):
    df_players = pd.read_csv(player_path, nrows=5000, low_memory=False)
    
    # Positional columns typically include tracking regions like 'st', 'ls', 'cm', 'cb'
    sample_pos_cols = ['ls', 'st', 'cm', 'cdm', 'cb']
    existing_sample_cols = [c for c in sample_pos_cols if c in df_players.columns]
    
    if existing_sample_cols:
        print(f"Auditing positional calculation patterns on columns: {existing_sample_cols}\n")
        
        for col in existing_sample_cols:
            # Drop null values to check raw values
            raw_series = df_players[col].dropna().astype(str)
            
            # Detect strings containing math operations '+' or '-'
            math_pattern_mask = raw_series.str.contains(r'\+|\-', regex=True)
            math_pattern_count = math_pattern_mask.sum()
            
            print(f"Column '{col}':")
            print(f"  Total Evaluated: {len(raw_series)}")
            print(f"  Contains modifications (e.g., '75+2'): {math_pattern_count} rows ({math_pattern_count/len(raw_series)*100:.1f}%)")
            if math_pattern_count > 0:
                print(f"  Sample raw variations: {raw_series[math_pattern_mask].unique()[:3]}")
    else:
        print("No tactical positional role tracking strings detected in this dataset sample.")

=== ANALYZING STRING-BASED MATHEMATICAL RATING ATTRIBUTES ===
Auditing positional calculation patterns on columns: ['ls', 'st', 'cm', 'cdm', 'cb']

Column 'ls':
  Total Evaluated: 5000
  Contains modifications (e.g., '75+2'): 4618 rows (92.4%)
  Sample raw variations: <StringArray>
['90+3', '83+3', '85+3']
Length: 3, dtype: str
Column 'st':
  Total Evaluated: 5000
  Contains modifications (e.g., '75+2'): 4618 rows (92.4%)
  Sample raw variations: <StringArray>
['90+3', '83+3', '85+3']
Length: 3, dtype: str
Column 'cm':
  Total Evaluated: 5000
  Contains modifications (e.g., '75+2'): 4671 rows (93.4%)
  Sample raw variations: <StringArray>
['81+3', '74+3', '90+1']
Length: 3, dtype: str
Column 'cdm':
  Total Evaluated: 5000
  Contains modifications (e.g., '75+2'): 4915 rows (98.3%)
  Sample raw variations: <StringArray>
['63+3', '80+3', '64+3']
Length: 3, dtype: str
Column 'cb':
  Total Evaluated: 5000
  Contains modifications (e.g., '75+2'): 4544 rows (90.9%)
  Sample raw variations: <S

In [7]:
print("=== RUNNING SPARSITY AND EMPTY METRIC PROFILES ===")

def calculate_sparsity(path, name):
    if not os.path.exists(path):
        return None
        
    df = pd.read_csv(path, low_memory=False)
    null_percentages = (df.isnull().sum() / len(df)) * 100
    
    # Isolate attributes where over 30% of records are missing
    highly_sparse_features = null_percentages[null_percentages > 30].sort_values(ascending=False)
    
    print(f"\nSparsity analysis for {name}:")
    print(f"  Total Features tracked: {len(df.columns)}")
    print(f"  Features with >30% missing entries: {len(highly_sparse_features)}")
    
    for col, pct in highly_sparse_features.head(5).items():
        print(f"    - {col}: {pct:.2f}% null")
        
    return pd.DataFrame({"Feature": highly_sparse_features.index, "Null_Percentage": highly_sparse_features.values, "Dataset": name})

sparsity_frames = [calculate_sparsity(path, name) for name, path in file_mappings.items()]

=== RUNNING SPARSITY AND EMPTY METRIC PROFILES ===

Sparsity analysis for male_players:
  Total Features tracked: 109
  Features with >30% missing entries: 8
    - nation_team_id: 94.39% null
    - nation_position: 94.39% null
    - nation_jersey_number: 94.39% null
    - club_loaned_from: 94.04% null
    - player_tags: 92.27% null

Sparsity analysis for female_players:
  Total Features tracked: 109
  Features with >30% missing entries: 19
    - club_loaned_from: 99.46% null
    - goalkeeping_speed: 87.77% null
    - player_tags: 86.89% null
    - player_traits: 59.34% null
    - club_joined_date: 55.95% null

Sparsity analysis for male_teams:
  Total Features tracked: 54
  Features with >30% missing entries: 22
    - off_style: 69.11% null
    - off_build_up_play: 69.08% null
    - off_chance_creation: 69.08% null
    - def_defence_width: 51.75% null
    - def_defence_aggression: 51.75% null

Sparsity analysis for female_teams:
  Total Features tracked: 54
  Features with >30% missing

In [8]:
print("=== AUDITING MULTI-VALUE DELIMITED STRINGS ===")

if os.path.exists(file_mappings["male_players"]):
    df_p = pd.read_csv(file_mappings["male_players"], nrows=10000, low_memory=False)
    
    target_delimited_cols = ['player_positions', 'player_tags', 'player_traits']
    
    for col in target_delimited_cols:
        if col in df_p.columns:
            non_null_data = df_p[col].dropna().astype(str)
            print(f"\nFeature: {col}")
            print(f"  Filled entries: {len(non_null_data)} / {len(df_p)}")
            print(f"  Sample values: {non_null_data.head(3).tolist()}")
            
            # Determine standard delimiter character (usually comma)
            all_delimiters = non_null_data.str.findall(r'[,|;]')
            flattened_delims = [item for sublist in all_delimiters for item in sublist]
            if flattened_delims:
                most_common_delim = max(set(flattened_delims), key=flattened_delims.count)
                print(f"  Detected standard separator element: '{most_common_delim}'")

=== AUDITING MULTI-VALUE DELIMITED STRINGS ===

Feature: player_positions
  Filled entries: 10000 / 10000
  Sample values: ['ST, LW', 'ST', 'CM, CAM']
  Detected standard separator element: ','

Feature: player_tags
  Filled entries: 1094 / 10000
  Sample values: ['#Speedster, #Dribbler, #Acrobat, #Clinical finisher, #Complete forward', '#Aerial threat, #Distance shooter, #Strength, #Clinical finisher, #Complete forward', '#Dribbler, #Playmaker, #Distance shooter, #Crosser, #Complete midfielder']
  Detected standard separator element: ','

Feature: player_traits
  Filled entries: 6523 / 10000
  Sample values: ['Quick Step +, Rapid, Flair, Trivela', 'Acrobatic +, Power Header, Quick Step', 'Pinged Pass +, Dead Ball, Incisive Pass, Long Ball Pass, Whipped Cross, Trivela']
  Detected standard separator element: ','


In [9]:
print("=== AUDITING CROSS-COLLECTION REFS ===")

if os.path.exists(file_mappings["male_players"]) and os.path.exists(file_mappings["male_teams"]):
    df_p = pd.read_csv(file_mappings["male_players"], nrows=10000, low_memory=False)
    df_t = pd.read_csv(file_mappings["male_teams"], low_memory=False)
    
    # Extract unique team identities present in reference collections
    master_team_ids = set(df_t['team_id'].dropna().unique())
    player_club_ids = set(df_p['club_team_id'].dropna().astype(int).unique())
    
    # Calculate cross-reference drops
    unmapped_teams = player_club_ids - master_team_ids
    
    print(f"Total Unique Team Profiles in Master Team list: {len(master_team_ids)}")
    print(f"Total Unique Club References inside Player Sample: {len(player_club_ids)}")
    print(f"Orphaned References (Teams in player data missing from team data): {len(unmapped_teams)}")
    if unmapped_teams:
        print(f"  Sample orphaned keys: {list(unmapped_teams)[:5]}")

=== AUDITING CROSS-COLLECTION REFS ===
Total Unique Team Profiles in Master Team list: 1145
Total Unique Club References inside Player Sample: 640
Orphaned References (Teams in player data missing from team data): 0
